# Диффузионная J-линза LLaDA-8B-Instruct

Линза строится на собственных траекториях расшумления LLaDA (раздел Method статьи).

1. Для 1000 текстов WikiText длиной не меньше 128 токенов перебираются длины промпта 16, 24, …, 120. Оставшиеся `128 − p` позиций генерируются сэмплером LLaDA одним блоком, по одному токену за шаг, с жадным перемаскированием по низкой уверенности; после каждого шага сохраняется снимок последовательности.
2. Источники — уже раскрытые позиции после первых 16, цели — все позиции ответа, раскрытые и ещё замаскированные.
3. Снимки делятся на 4 бина по доле замаскированных токенов ответа; из каждого бина случайно берётся 2500 снимков (всего 10 000).
4. На каждом бине обучается своя линза, затем линзы бинов объединяются (`JacobianLens.merge`, веса по числу примеров) в основную линзу `merged`.

Выход: `new_llada_diffusion_lens1000_bin{0..3}.pt` и `new_llada_diffusion_lens_merged_1000.pt`.


In [1]:
import torch
import transformers

import jlens
jlens.configure_logging()

from jlens import Layout
from jlens.lens import JacobianLens
from jlens.examples import load_wikitext_prompts

from jlens.diffusion_corpus import (
    build_diffusion_examples, MASK_ID, ILLADA_MASK_ID, get_mask_token_id,
)

import os

In [2]:
import numpy as np
import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
device = "cuda:0"

# GSAI-ML/LLaDA-8B-Instruct

У LLaDA нестандартная раскладка модулей, поэтому нужна обёртка `WrappedLLaDA` и свой `Layout`: блоки — `transformer.blocks`, финальная нормализация — `transformer.ln_f`, разэмбеддинг — `transformer.ff_out`. Идентификатор маски читается из конфига модели (`mask_token_id = 126336`).


In [4]:
model_name = "GSAI-ML/LLaDA-8B-Instruct"

In [5]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
hf_model = transformers.AutoModel.from_pretrained(
    model_name, trust_remote_code=True, torch_dtype=torch.bfloat16,
).to(device)

hf_model

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 6/6 [00:00<00:00, 190.35it/s]


LLaDAModelLM(
  (model): LLaDAModel(
    (transformer): ModuleDict(
      (wte): Embedding(126464, 4096)
      (emb_drop): Dropout(p=0.0, inplace=False)
      (ln_f): RMSLayerNorm()
      (blocks): ModuleList(
        (0-31): 32 x LLaDALlamaBlock(
          (dropout): Dropout(p=0.0, inplace=False)
          (act): SiLU()
          (attn_out): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_out): Linear(in_features=12288, out_features=4096, bias=False)
          (rotary_emb): RotaryEmbedding()
          (attn_norm): RMSLayerNorm()
          (ff_norm): RMSLayerNorm()
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        )
      )
    

In [6]:
class WrappedLLaDA:
    def __init__(self, model):
        self.model = model
        self.blocks = model.transformer.blocks
        self.ln_f = model.transformer.ln_f
        self.wte = model.transformer.wte

    def forward(self, input_ids, **kwargs):
        return self.model(input_ids, **kwargs)

    def __call__(self, input_ids, **kwargs):
        return self.forward(input_ids, **kwargs)

hf_model.text_module = WrappedLLaDA(hf_model.model)
hf_model.lm_head = hf_model.model.transformer.ff_out

layout = Layout(
    path="text_module",
    layers="blocks",
    norm="ln_f",
    embed="wte",
    lm_head="lm_head"
)

model = jlens.from_hf(hf_model, tokenizer, layout=layout)
model

HFLensModel(LLaDAModelLM, n_layers=32, d_model=4096)

## Корпус траекторий

Сбор корпуса — самый долгий шаг. Готовый корпус сохраняется в `llada_diffusion_corpus.pkl` и при повторном запуске загружается с диска.


In [ ]:
import pickle
from pathlib import Path

corpus_path = Path("./llada_diffusion_corpus.pkl")
mask_id = get_mask_token_id(hf_model, fallback=MASK_ID)
print(f"mask_id = {mask_id}")

In [ ]:
if not corpus_path.exists():
    prompt_texts = load_wikitext_prompts(n_prompts=1000)
    examples, source_masks, target_masks, tau_bins = build_diffusion_examples(
        hf_model, tokenizer, prompt_texts, device=device,
        total_len=128,
        prompt_lens=range(16, 128, 8),
        skip_first_source=16,
        n_bins=4,
        mask_id=mask_id,
    )
    with open(corpus_path, "wb") as f:
        pickle.dump(
            {
                "examples": [e.cpu() for e in examples],
                "source_masks": source_masks,
                "target_masks": target_masks,
                "tau_bins": tau_bins,
            },
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    print(f"saved {len(examples)} snapshots -> {corpus_path}")

In [10]:
with open(corpus_path, "rb") as f:
    corpus = pickle.load(f)

examples = corpus["examples"]
source_masks = corpus["source_masks"]
target_masks = corpus["target_masks"]
tau_bins = corpus["tau_bins"]

print(f"Загружено {len(examples)} снэпшотов из {corpus_path}")

Загружено 822360 снэпшотов из llada_diffusion_corpus.pkl


In [11]:
import random
from collections import Counter, defaultdict

TARGET_TOTAL = 10000

bin_to_indices = defaultdict(list)
for i, b in enumerate(tau_bins):
    bin_to_indices[b].append(i)

n_bins_present = len(bin_to_indices)
per_bin_target = TARGET_TOTAL // n_bins_present

rng = random.Random(seed)

sampled_idx = []
for b, idx_list in sorted(bin_to_indices.items()):
    k = min(per_bin_target, len(idx_list))
    if k < per_bin_target:
        print(f"ВНИМАНИЕ: бин {b} содержит только {len(idx_list)} снэпшотов "
              f"(запрошено {per_bin_target}) -- берём все имеющиеся")
    sampled_idx.extend(rng.sample(idx_list, k))

rng.shuffle(sampled_idx)

examples_sub = [examples[i].to(device)  for i in sampled_idx]
source_masks_sub = [source_masks[i] for i in sampled_idx]
target_masks_sub = [target_masks[i] for i in sampled_idx]
tau_bins_sub = [tau_bins[i] for i in sampled_idx]

print(f"Собрано всего: {len(examples)} снэпшотов")
print(f"После субсэмплирования: {len(examples_sub)} снэпшотов")
print("распределение по бинам:", dict(sorted(Counter(tau_bins_sub).items())))

Собрано всего: 822360 снэпшотов
После субсэмплирования: 10000 снэпшотов
распределение по бинам: {0: 2500, 1: 2500, 2: 2500, 3: 2500}


## Линзы бинов и линза merged

На каждом бине обучается отдельная линза; `merged` — их среднее с весами по числу примеров (`JacobianLens.merge`), то есть линза, обученная на всём корпусе сразу.


In [12]:
sorted(set(tau_bins))

[0, 1, 2, 3]

In [ ]:
bin_lenses = {}
for b in list(sorted(set(tau_bins_sub))):
    idx = [i for i, bb in enumerate(tau_bins_sub) if bb == b]
    print(f"--- бин tau={b}: {len(idx)} снэпшотов ---")
    bin_lenses[b] = jlens.fitting.fit(
        model,
        [examples_sub[i] for i in idx],
        source_position_masks=[source_masks_sub[i] for i in idx],
        target_position_masks=[target_masks_sub[i] for i in idx],
        dim_batch=64,
        skip_first=16,
        checkpoint_path=f"./new_llada_diffusion_ckpt1000_bin{b}.pt",
        checkpoint_every=50
    )
    bin_lenses[b].save(f"./new_llada_diffusion_lens1000_bin{b}.pt")
    print(bin_lenses[b])

[  2m24s +144.90s] fit: n_layers=32 d_model=4096, fitting 31 source layers (target=L31) on 2500 examples


--- бин tau=0: 2500 снэпшотов ---


[  2m28s +  3.58s]   resuming from checkpoint: 200/2500 examples processed
[  2m52s + 23.85s]   example 201/2500  seq_len=128 n_valid=107  24s  max||J||/sqrt(d)=8.673  max_d_mean=4.91e-02
[  3m15s + 23.59s]   example 202/2500  seq_len=128 n_valid=109  24s  max||J||/sqrt(d)=1.058  max_d_mean=8.05e-03
[  3m39s + 23.90s]   example 203/2500  seq_len=128 n_valid=102  24s  max||J||/sqrt(d)=10.541  max_d_mean=6.10e-02
[  4m03s + 23.91s]   example 204/2500  seq_len=128 n_valid=108  24s  max||J||/sqrt(d)=4.062  max_d_mean=2.59e-02
[  4m27s + 23.60s]   example 205/2500  seq_len=128 n_valid=106  24s  max||J||/sqrt(d)=2.697  max_d_mean=1.77e-02
[  4m51s + 24.23s]   example 206/2500  seq_len=128 n_valid=111  24s  max||J||/sqrt(d)=5.876  max_d_mean=3.37e-02
[  5m15s + 23.80s]   example 207/2500  seq_len=128 n_valid=101  24s  max||J||/sqrt(d)=6.306  max_d_mean=4.31e-02
[  5m38s + 23.30s]   example 208/2500  seq_len=128 n_valid=101  23s  max||J||/sqrt(d)=6.796  max_d_mean=3.82e-02
[  6m02s + 23.79s]  

In [ ]:
overall_lens = JacobianLens.merge(list(bin_lenses.values()))
overall_lens.save("./new_llada_diffusion_lens_merged_1000.pt")
overall_lens